# Tennis Ball Tracking with TrackNet - GPU Version
Pick a clip from your Google Drive, track the ball on GPU, and save results back to Drive.

## Setup
1. **Runtime → Change Runtime Type → GPU**
2. Run the cells in order. When prompted, mount the Google Drive account that
   holds your clips (the one with `tennis_motion/clips/output/...`).
3. A **dropdown** lists every clip found — pick one, then run the rest.

## Notes
- **Streaming, constant memory:** the video is processed one frame at a time
  (read → detect → annotate → write), so even long 1080p clips won't blow up RAM.
  A standard (non-high-RAM) GPU runtime is fine.
- The TrackNet model is **auto-downloaded** — you don't need to store it in Drive.
- Frames are auto-resized to **1280×720** (TrackNet's expected input).
- Outputs are saved **next to the source clip** on Drive:
  - `<clip>_balltracked.mp4` — annotated video
  - `<clip>_ball_tracking.csv` / `.json` — ball positions per frame

In [ ]:
# Install dependencies
!pip install opencv-python scipy torch torchvision tqdm gdown ipywidgets -q
print("Dependencies installed!")

In [ ]:
import cv2
import torch
import numpy as np
import os, glob, shutil
from scipy.spatial import distance
from tqdm import tqdm
from google.colab import drive
import ipywidgets as widgets
from IPython.display import display

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

In [ ]:
# Mount Google Drive (use the account that holds your tennis_motion clips)
drive.mount('/content/drive')

# Folder that contains your clips, organized as output/<id>/<id>_trimmed.mp4
ROOT = '/content/drive/My Drive/tennis_motion/clips/output'

assert os.path.isdir(ROOT), (
    f"Folder not found:\n  {ROOT}\n"
    "Mount the Drive for the account that has tennis_motion, or edit ROOT above."
)
print('Clips root:', ROOT)

In [ ]:
# Define TrackNet Model Architecture
import torch.nn as nn

class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, pad=1, stride=1, bias=True):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride, padding=pad, bias=bias),
            nn.ReLU(),
            nn.BatchNorm2d(out_channels)
        )

    def forward(self, x):
        return self.block(x)

class BallTrackerNet(nn.Module):
    def __init__(self, input_channels=3, out_channels=14):
        super().__init__()
        self.out_channels = out_channels
        self.input_channels = input_channels

        self.conv1 = ConvBlock(in_channels=self.input_channels, out_channels=64)
        self.conv2 = ConvBlock(in_channels=64, out_channels=64)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv3 = ConvBlock(in_channels=64, out_channels=128)
        self.conv4 = ConvBlock(in_channels=128, out_channels=128)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv5 = ConvBlock(in_channels=128, out_channels=256)
        self.conv6 = ConvBlock(in_channels=256, out_channels=256)
        self.conv7 = ConvBlock(in_channels=256, out_channels=256)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv8 = ConvBlock(in_channels=256, out_channels=512)
        self.conv9 = ConvBlock(in_channels=512, out_channels=512)
        self.conv10 = ConvBlock(in_channels=512, out_channels=512)
        self.ups1 = nn.Upsample(scale_factor=2)
        self.conv11 = ConvBlock(in_channels=512, out_channels=256)
        self.conv12 = ConvBlock(in_channels=256, out_channels=256)
        self.conv13 = ConvBlock(in_channels=256, out_channels=256)
        self.ups2 = nn.Upsample(scale_factor=2)
        self.conv14 = ConvBlock(in_channels=256, out_channels=128)
        self.conv15 = ConvBlock(in_channels=128, out_channels=128)
        self.ups3 = nn.Upsample(scale_factor=2)
        self.conv16 = ConvBlock(in_channels=128, out_channels=64)
        self.conv17 = ConvBlock(in_channels=64, out_channels=64)
        self.conv18 = ConvBlock(in_channels=64, out_channels=self.out_channels)

        self._init_weights()

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.pool1(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.pool2(x)
        x = self.conv5(x)
        x = self.conv6(x)
        x = self.conv7(x)
        x = self.pool3(x)
        x = self.conv8(x)
        x = self.conv9(x)
        x = self.conv10(x)
        x = self.ups1(x)
        x = self.conv11(x)
        x = self.conv12(x)
        x = self.conv13(x)
        x = self.ups2(x)
        x = self.conv14(x)
        x = self.conv15(x)
        x = self.ups3(x)
        x = self.conv16(x)
        x = self.conv17(x)
        x = self.conv18(x)
        return x

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.uniform_(module.weight, -0.05, 0.05)
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0)
            elif isinstance(module, nn.BatchNorm2d):
                nn.init.constant_(module.weight, 1)
                nn.init.constant_(module.bias, 0)

print("TrackNet model defined")

In [ ]:
# Define Ball Detector
class BallDetector:
    def __init__(self, path_model, device='cuda'):
        self.model = BallTrackerNet(input_channels=9, out_channels=256)
        self.device = device
        self.model.load_state_dict(torch.load(path_model, map_location=device))
        self.model = self.model.to(device)
        self.model.eval()
        self.width = 640
        self.height = 360
        print(f'Model loaded on {device}')

    def infer_model(self, frames):
        ball_track = [(None, None)] * 2
        prev_pred = [None, None]
        for num in tqdm(range(2, len(frames)), desc='Ball Detection'):
            img = cv2.resize(frames[num], (self.width, self.height))
            img_prev = cv2.resize(frames[num-1], (self.width, self.height))
            img_preprev = cv2.resize(frames[num-2], (self.width, self.height))
            imgs = np.concatenate((img, img_prev, img_preprev), axis=2)
            imgs = imgs.astype(np.float32) / 255.0
            imgs = np.rollaxis(imgs, 2, 0)
            inp = np.expand_dims(imgs, axis=0)

            with torch.no_grad():
                out = self.model(torch.from_numpy(inp).float().to(self.device))
            output = out.argmax(dim=1).detach().cpu().numpy()
            x_pred, y_pred = self.postprocess(output, prev_pred)
            prev_pred = [x_pred, y_pred]
            ball_track.append((x_pred, y_pred))
        return ball_track

    def postprocess(self, feature_map, prev_pred, scale=2, max_dist=80):
        feature_map *= 255
        feature_map = feature_map.reshape((self.height, self.width))
        feature_map = feature_map.astype(np.uint8)
        ret, heatmap = cv2.threshold(feature_map, 127, 255, cv2.THRESH_BINARY)
        circles = cv2.HoughCircles(heatmap, cv2.HOUGH_GRADIENT, dp=1, minDist=1,
                                   param1=50, param2=2, minRadius=2, maxRadius=7)
        x, y = None, None
        if circles is not None:
            if prev_pred[0]:
                for i in range(len(circles[0])):
                    x_temp = circles[0][i][0] * scale
                    y_temp = circles[0][i][1] * scale
                    dist = distance.euclidean((x_temp, y_temp), prev_pred)
                    if dist < max_dist:
                        x, y = x_temp, y_temp
                        break
            else:
                x = circles[0][0][0] * scale
                y = circles[0][0][1] * scale
        return x, y

print("BallDetector class defined")

In [ ]:
# Streaming processor — constant memory, handles long / 1080p videos
TARGET_W, TARGET_H = 1280, 720  # TrackNet expects 1280x720 for correct ball coords

def process_video_streaming(input_path, output_path, detector, target=(TARGET_W, TARGET_H)):
    """Single pass: read -> infer -> annotate -> write, one frame at a time.
    Never holds the whole video in RAM. Returns (ball_track, fps)."""
    W, H = target
    cap = cv2.VideoCapture(input_path)
    fps = int(cap.get(cv2.CAP_PROP_FPS)) or 25
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (W, H))

    ball_track = []
    buf = []            # rolling buffer of last <=3 frames (resized to WxH)
    prev_pred = [None, None]
    pbar = tqdm(total=total if total > 0 else None, desc='Processing')

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if (frame.shape[1], frame.shape[0]) != (W, H):
            frame = cv2.resize(frame, (W, H))
        buf.append(frame)
        if len(buf) > 3:
            buf.pop(0)

        x_pred, y_pred = None, None
        if len(buf) == 3:
            # model input = current, prev, preprev (matches original ordering)
            trio = [cv2.resize(b, (detector.width, detector.height)) for b in (buf[-1], buf[-2], buf[-3])]
            imgs = np.concatenate(trio, axis=2).astype(np.float32) / 255.0
            imgs = np.rollaxis(imgs, 2, 0)[None]
            with torch.no_grad():
                o = detector.model(torch.from_numpy(imgs).float().to(detector.device))
            o = o.argmax(dim=1).detach().cpu().numpy()
            x_pred, y_pred = detector.postprocess(o, prev_pred)
            prev_pred = [x_pred, y_pred]

        ball_track.append((x_pred, y_pred))

        # annotate current frame and write straight to disk
        img_res = buf[-1]
        if x_pred is not None:
            xi, yi = int(x_pred), int(y_pred)
            cv2.circle(img_res, (xi, yi), radius=5, color=(0, 255, 0), thickness=2)
            cv2.putText(img_res, 'ball', org=(xi + 8, yi + 8),
                        fontFace=cv2.FONT_HERSHEY_SIMPLEX, fontScale=0.8,
                        thickness=2, color=(0, 255, 0))
        out.write(img_res)
        pbar.update(1)

    pbar.close()
    cap.release()
    out.release()
    return ball_track, fps

print("Streaming processor defined")

In [ ]:
# Discover all clips under ROOT and choose one to process
all_videos = sorted(glob.glob(os.path.join(ROOT, '**', '*.mp4'), recursive=True))
# hide previously generated outputs
all_videos = [v for v in all_videos if '_balltracked' not in os.path.basename(v)]
assert all_videos, f'No .mp4 files found under {ROOT}'

print(f'Found {len(all_videos)} clip(s).')
options = [(os.path.relpath(v, ROOT), v) for v in all_videos]
video_selector = widgets.Dropdown(options=options, description='Clip:',
                                   layout=widgets.Layout(width='90%'))
display(video_selector)
print('Pick a clip above, then run the next cell.')

In [ ]:
# Resolve model + detector + select clip (no full-video load -> low RAM)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

# --- TrackNet weights: auto-download once to local disk ---
model_path = '/content/tracknet_model.pt'
if not os.path.exists(model_path):
    import gdown
    print('Downloading TrackNet model...')
    gdown.download(id='1XEYZ4myUN7QT-NeBYJI0xteLsvs-ZAOl', output=model_path, quiet=False)

detector = BallDetector(model_path, device)

# --- Selected clip from the dropdown ---
drive_video = video_selector.value
print('Selected clip:', os.path.relpath(drive_video, ROOT))

# Copy from Drive to fast local disk (we still stream-read from local)
local_video = '/content/input_video.mp4'
print('Copying clip to local disk...')
shutil.copy(drive_video, local_video)
print(f'Copied ({os.path.getsize(local_video)/1e6:.1f} MB)')

# Metadata only (instant, no frame load)
cap = cv2.VideoCapture(local_video)
src_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); src_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
src_n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); src_fps = cap.get(cv2.CAP_PROP_FPS)
cap.release()
print(f'Source: {src_w}x{src_h}, {src_n} frames @ {src_fps:.1f} fps  (~{src_n/(src_fps or 25)/60:.1f} min)')
if (src_w, src_h) != (TARGET_W, TARGET_H):
    print(f'  -> frames resized to {TARGET_W}x{TARGET_H} for TrackNet')

# Output paths next to the source clip on Drive
src_dir = os.path.dirname(drive_video)
base = os.path.splitext(os.path.basename(drive_video))[0]
local_output = f'/content/{base}_balltracked.mp4'
drive_output = os.path.join(src_dir, f'{base}_balltracked.mp4')
print('Ready to process.')

In [ ]:
# Process the clip in one streaming pass (detect + annotate + write)
print('Streaming through the video...')
ball_track, fps = process_video_streaming(local_video, local_output, detector)

detected_count = sum(1 for x, y in ball_track if x is not None)
print(f"\nDone. Ball detected in {detected_count}/{len(ball_track)} frames "
      f"({100*detected_count/max(len(ball_track),1):.1f}%)")
print(f"Annotated video written locally: {local_output} "
      f"({os.path.getsize(local_output)/1e6:.1f} MB)")

In [ ]:
# Save the annotated video to Drive (next to the source clip)
shutil.copy(local_output, drive_output)
print(f"Saved annotated video to Drive:\n  {drive_output}")
print(f"File size: {os.path.getsize(drive_output) / 1e6:.1f} MB")

In [ ]:
# (Optional) also download the annotated video to your computer
# Everything is already saved to your Drive next to the source clip.
from google.colab import files
files.download(local_output)

In [ ]:
# Save ball positions next to the source clip on Drive (CSV + JSON)
import csv, json

# src_dir and base come from the video-generation cell above
csv_path  = os.path.join(src_dir, f'{base}_ball_tracking.csv')
json_path = os.path.join(src_dir, f'{base}_ball_tracking.json')

# Coordinates are in the 1280x720 space the model ran on.
# Also provide normalized [0,1] coords so they map onto ANY resolution.
rows = []
for i, (x, y) in enumerate(ball_track):
    visible = x is not None and y is not None
    rows.append({
        'frame': i,
        'time_sec': round(i / fps, 4) if fps else None,
        'x': round(float(x), 2) if visible else '',          # pixels @ 1280x720
        'y': round(float(y), 2) if visible else '',
        'x_norm': round(float(x) / TARGET_W, 5) if visible else '',  # 0..1
        'y_norm': round(float(y) / TARGET_H, 5) if visible else '',
        'visible': int(visible),
    })

# --- CSV (one row per frame; empty x/y when ball not detected) ---
with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['frame', 'time_sec', 'x', 'y', 'x_norm', 'y_norm', 'visible'])
    writer.writeheader()
    writer.writerows(rows)

# --- JSON (metadata + data, easy to load in another project) ---
detected = sum(r['visible'] for r in rows)
payload = {
    'source_video': os.path.basename(drive_video),
    'fps': fps,
    'num_frames': len(ball_track),
    'frames_with_ball': detected,
    'detection_rate': round(detected / len(ball_track), 4) if ball_track else 0,
    'coordinate_space': {'width': TARGET_W, 'height': TARGET_H,
                         'note': 'x,y are pixels at 1280x720; x_norm,y_norm are 0..1 for any resolution'},
    'data': [{'frame': r['frame'], 'time_sec': r['time_sec'],
              'x': (r['x'] if r['visible'] else None),
              'y': (r['y'] if r['visible'] else None),
              'visible': bool(r['visible'])} for r in rows],
}
with open(json_path, 'w') as f:
    json.dump(payload, f, indent=2)

print('Saved to Drive (next to the source clip):')
print(' ', csv_path)
print(' ', json_path)
print(f'Ball detected in {detected}/{len(ball_track)} frames ({payload["detection_rate"]*100:.1f}%)')

# (Optional) also download the spreadsheets to your computer
from google.colab import files
files.download(csv_path)
files.download(json_path)